# Tomato Leaf Disease Detection - Final Inference Pipeline with Calibration and TTA

This notebook documents the final project stage: inference, confidence calibration, test-time augmentation, confidence-based accept/reject logic, final evaluation, and mobile application integration notes.

**Portfolio note:** This notebook is part of a cleaned public portfolio version. Large datasets, trained model weights, HPC runtime artifacts, and private machine paths are not included.

**Public repository sections:**

- Final Inference Pipeline
- Image Preprocessing
- Disease Classification
- Confidence Calibration
- Test-Time Augmentation
- Accept/Reject Logic
- Final Evaluation Results
- Mobile Application Integration
- Deployment Notes



# Track B Experiment 3 - Final Pipeline Presentation

This notebook is the presentation version of **Experiment 3**.
It reuses the saved outputs from the original run and focuses on the final system story:

- stronger EfficientNet-B3 training strategy
- EMA model selection
- calibrated TTA inference
- reject / uncertainty logic
- comparison against Experiment 2
- final deployment-style evidence



## Section 0 - Paths And Presentation Save Folder


In [ ]:

from pathlib import Path

PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'Universal_Tomato_Dataset'
SOURCE_EXPERIMENT3_ROOT = PROJECT_ROOT / 'trackB_outputs' / 'Track_B_Experiment_3'
SOURCE_EXPERIMENT2_ROOT = PROJECT_ROOT / 'trackB_outputs' / 'Track_B_Experiment_2'
PRESENTATION_ROOT = PROJECT_ROOT / 'trackB_outputs' / 'Track_B_Experiment_3_Presentation'

SAVE_DIRS = {
    'root': PRESENTATION_ROOT,
    'figures': PRESENTATION_ROOT / 'figures',
    'tables': PRESENTATION_ROOT / 'tables',
}

for path in SAVE_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

required_paths = [
    SOURCE_EXPERIMENT3_ROOT,
    SOURCE_EXPERIMENT3_ROOT / 'deployment' / 'final_experiment3_summary.json',
    SOURCE_EXPERIMENT3_ROOT / 'disease_classifier' / 'base_vs_ema_summary.json',
    SOURCE_EXPERIMENT3_ROOT / 'disease_classifier' / 'full_dataset_tta_summary.json',
    SOURCE_EXPERIMENT3_ROOT / 'thresholds' / 'accepted_only_test_metrics_v3.json',
    SOURCE_EXPERIMENT3_ROOT / 'comparisons' / 'experiment2_vs_experiment3_summary.csv',
]

for req in required_paths:
    if not req.exists():
        raise FileNotFoundError(f'Missing required file or folder: {req}')

print('Experiment 3 source root      :', SOURCE_EXPERIMENT3_ROOT)
print('Experiment 3 presentation root:', PRESENTATION_ROOT)
print('Experiment 2 reference root   :', SOURCE_EXPERIMENT2_ROOT)



## Section 1 - Imports, Theme, And Helpers


In [ ]:

import json
import math
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

PRESENTATION_COLORS = {
    'navy': '#12355B',
    'teal': '#1B998B',
    'gold': '#F4A259',
    'red': '#BC4749',
    'green': '#4C956C',
    'gray': '#6C757D',
    'light': '#F7FAFC',
    'blue': '#2D6A9F',
}

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10


def read_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def percent(x: float) -> float:
    return round(float(x) * 100, 2)


def display_label(name: str) -> str:
    return str(name).replace('_', ' ')


def human_readable_int(x: int) -> str:
    return f'{int(x):,}'


def save_styled_table_figure(
    df: pd.DataFrame,
    path: Path,
    title: str,
    figsize=(14, 6),
    font_size: int = 10,
) -> None:
    table_df = df.copy().fillna('').astype(str)

    max_col_lengths = {
        col: max([len(str(col))] + table_df[col].map(len).tolist())
        for col in table_df.columns
    }
    total_col_length = max(sum(max_col_lengths.values()), 1)
    total_wrap_budget = max(20 * len(table_df.columns), 84)

    wrap_widths = {
        col: max(
            16,
            min(42, round(total_wrap_budget * max_col_lengths[col] / total_col_length)),
        )
        for col in table_df.columns
    }

    wrapped_df = table_df.copy()
    for col in wrapped_df.columns:
        width = wrap_widths[col]
        wrapped_df[col] = wrapped_df[col].map(
            lambda x, width=width: textwrap.fill(
                x,
                width=width,
                break_long_words=False,
                break_on_hyphens=False,
            )
        )

    wrapped_columns = [
        textwrap.fill(str(col), width=max(14, wrap_widths[col]))
        for col in wrapped_df.columns
    ]

    row_line_counts = wrapped_df.apply(
        lambda row: max(max(1, len(str(value).splitlines())) for value in row),
        axis=1,
    )

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.set_title(title, pad=16, fontsize=16, fontweight='bold')

    col_widths = np.array(
        [
            max(max_col_lengths[col], np.mean(list(max_col_lengths.values())) * 0.65)
            for col in wrapped_df.columns
        ],
        dtype=float,
    )
    col_widths = (col_widths / col_widths.sum()).tolist()

    table = ax.table(
        cellText=wrapped_df.values,
        colLabels=wrapped_columns,
        cellLoc='left',
        colLoc='left',
        colWidths=col_widths,
        bbox=[0, 0, 1, 0.86],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    header_weight = 1.2
    row_weights = [max(1.0, 0.9 + 0.45 * (line_count - 1)) for line_count in row_line_counts]
    total_height_weight = header_weight + sum(row_weights)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#25364A')
        cell.set_linewidth(0.9)
        cell.PAD = 0.10
        cell.get_text().set_ha('left')
        cell.get_text().set_va('center')

        if row == 0:
            cell.set_facecolor(PRESENTATION_COLORS['navy'])
            cell.set_text_props(color='white', weight='bold')
            cell.set_height(0.86 * header_weight / total_height_weight)
        else:
            cell.set_height(0.86 * row_weights[row - 1] / total_height_weight)
            cell.set_facecolor('white' if row % 2 == 1 else PRESENTATION_COLORS['light'])

    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight', facecolor='white')
    plt.show()



## Section 2 - Load Saved Experiment 3 Results


In [ ]:

summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'deployment' / 'final_experiment3_summary.json')
base_vs_ema_summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'disease_classifier' / 'base_vs_ema_summary.json')
tta_summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'disease_classifier' / 'full_dataset_tta_summary.json')
accepted_summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'thresholds' / 'accepted_only_test_metrics_v3.json')
leaf_gate_summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'leaf_gate' / 'leaf_gate_summary.json')
threshold_summary = read_json(SOURCE_EXPERIMENT3_ROOT / 'thresholds' / 'disease_threshold_summary_v3.json')

exp2_vs_exp3_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'comparisons' / 'experiment2_vs_experiment3_summary.csv')
per_class_compare_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'comparisons' / 'test_per_class_accuracy_vs_experiment2.csv')
deployment_preview_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'predictions' / 'experiment3_test_subset_deployment_predictions.csv')
disease_history_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'disease_classifier' / 'disease_history_v3.csv')
leaf_gate_history_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'leaf_gate' / 'leaf_gate_history.csv')
calibrated_per_class_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'reports' / 'selected_calibrated_tta_test_per_class_accuracy.csv')
hardest_confusions_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'reports' / 'selected_calibrated_tta_test_hardest_confusions.csv')
split_shift_df = pd.read_csv(SOURCE_EXPERIMENT3_ROOT / 'audits' / 'split_shift_summary.csv')
class_names = read_json(SOURCE_EXPERIMENT3_ROOT / 'deployment' / 'class_names.json')
final_confusion_matrix = np.load(SOURCE_EXPERIMENT3_ROOT / 'reports' / 'selected_calibrated_tta_test_confusion_matrix.npy')

selected_model_name = summary['selected_model_for_inference'].upper()

print('Experiment 3 presentation notebook is ready.')
print('Selected inference model    :', selected_model_name)
print('Disease backbone            :', summary['disease_model_name'])
print('Leaf gate backbone          :', summary['leaf_gate_model_name'])
print('Best epoch                  :', summary['training_summary']['best_epoch'])
print('Base test macro-F1          :', f"{percent(summary['selected_base_test_macro_f1']):.2f}%")
print('Calibrated TTA test macro-F1:', f"{percent(summary['calibrated_tta_test_macro_f1']):.2f}%")
print('Accepted-only macro-F1      :', f"{percent(summary['accepted_only_test_macro_f1']):.2f}%")
print('Experiment 3 vs Experiment 2:', f"+{percent(summary['experiment2_comparison']['delta_calibrated_tta_test_macro_f1_vs_exp2']):.2f} pts macro-F1")
print('Presentation output root    :', PRESENTATION_ROOT)



## Section 3 - Final Pipeline Overview


In [ ]:
fig, ax = plt.subplots(figsize=(14.5, 3.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

boxes = [
    (0.03, 'Input tomato image', PRESENTATION_COLORS['blue']),
    (0.21, 'Leaf gate\n(EfficientNet-B0)', PRESENTATION_COLORS['teal']),
    (0.40, 'Disease model\n(EfficientNet-B3 + EMA)', PRESENTATION_COLORS['navy']),
    (0.61, 'Temperature scaling\n+ 7-view TTA', PRESENTATION_COLORS['gold']),
    (0.79, 'Reject logic\n(confidence + margin)', PRESENTATION_COLORS['red']),
]

for x, label, color in boxes:
    rect = patches.FancyBboxPatch(
        (x, 0.34), 0.15, 0.30,
        boxstyle='round,pad=0.02,rounding_size=0.02',
        linewidth=1.2,
        edgecolor='#1F2F3F',
        facecolor=color,
        alpha=0.95,
    )
    ax.add_patch(rect)
    ax.text(
        x + 0.075, 0.49, label,
        ha='center', va='center',
        color='white', fontsize=11, fontweight='bold'
    )

for x1, x2 in [(0.18, 0.21), (0.36, 0.40), (0.55, 0.61), (0.76, 0.79)]:
    ax.annotate(
        '',
        xy=(x2, 0.49),
        xytext=(x1, 0.49),
        arrowprops=dict(arrowstyle='->', lw=2, color='#2F3E46'),
    )

accept_rect = patches.FancyBboxPatch(
    (0.90, 0.54), 0.08, 0.18,
    boxstyle='round,pad=0.02,rounding_size=0.02',
    linewidth=1.1,
    edgecolor='#1F2F3F',
    facecolor=PRESENTATION_COLORS['green'],
)
reject_rect = patches.FancyBboxPatch(
    (0.90, 0.24), 0.08, 0.18,
    boxstyle='round,pad=0.02,rounding_size=0.02',
    linewidth=1.1,
    edgecolor='#1F2F3F',
    facecolor=PRESENTATION_COLORS['gray'],
)

ax.add_patch(accept_rect)
ax.add_patch(reject_rect)

ax.text(
    0.94, 0.63, 'Accepted\nprediction',
    ha='center', va='center',
    color='white', fontsize=10.5, fontweight='bold'
)
ax.text(
    0.94, 0.33, 'Rejected\nuncertain case',
    ha='center', va='center',
    color='white', fontsize=10.5, fontweight='bold'
)

ax.annotate(
    '',
    xy=(0.90, 0.61),
    xytext=(0.94, 0.49),
    arrowprops=dict(arrowstyle='->', lw=1.8, color='#2F3E46'),
)
ax.annotate(
    '',
    xy=(0.90, 0.33),
    xytext=(0.94, 0.49),
    arrowprops=dict(arrowstyle='->', lw=1.8, color='#2F3E46'),
)

plt.title('Experiment 3 Final Inference Pipeline', pad=16)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_pipeline_overview.png', dpi=220, bbox_inches='tight')
plt.show()

component_df = pd.DataFrame([
    {
        'Component': 'Two-phase training',
        'Role': 'Phase A uses strong regularization, Phase B fine-tunes the final model.',
        'Why it matters': 'Improves generalization while still letting the model settle late in training.',
    },
    {
        'Component': 'EMA model selection',
        'Role': 'Tracks a smoothed copy of the disease model weights.',
        'Why it matters': 'Gives a more stable validation selection target than raw checkpoints.',
    },
    {
        'Component': 'Temperature scaling',
        'Role': 'Recalibrates logits on the validation set.',
        'Why it matters': 'Makes confidence values safer for threshold-based rejection.',
    },
    {
        'Component': '7-view TTA',
        'Role': 'Averages predictions across flips, zoom, brightness, darkness, and rotations.',
        'Why it matters': 'Raises final robustness on the held-out test set.',
    },
    {
        'Component': 'Reject logic',
        'Role': 'Uses confidence and top-2 margin thresholds before accepting a prediction.',
        'Why it matters': 'Avoids forcing low-confidence disease labels at deployment time.',
    },
    {
        'Component': 'Leaf gate',
        'Role': 'Checks whether a tomato leaf is present before disease classification.',
        'Why it matters': 'Acts as an early filter, although this version still uses synthetic negatives only.',
    },
])

component_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_component_summary.csv', index=False)

save_styled_table_figure(
    component_df,
    SAVE_DIRS['figures'] / 'experiment3_component_summary.png',
    title='Experiment 3 Design Choices',
    figsize=(17, 6.2),
    font_size=10,
)


## Section 4 - Training Strategy And Feature Engineering


In [ ]:

training_design_df = pd.DataFrame([
    {
        'Technique': 'Local pretrained weights',
        'Used in Experiment 3': 'Yes',
        'Purpose': 'Starts both backbones from stronger cached initialization.',
    },
    {
        'Technique': 'MixUp + CutMix',
        'Used in Experiment 3': 'Yes',
        'Purpose': 'Regularizes disease training during the first phase.',
    },
    {
        'Technique': 'Two-phase schedule',
        'Used in Experiment 3': 'Yes',
        'Purpose': 'Turns off mix augmentation near the end and fine-tunes the classifier.',
    },
    {
        'Technique': 'EMA tracking',
        'Used in Experiment 3': 'Yes',
        'Purpose': 'Smooths model weights and improves validation-based model selection.',
    },
    {
        'Technique': 'Class-weighted fine-tune loss',
        'Used in Experiment 3': 'Yes',
        'Purpose': 'Gives extra attention to smaller disease classes late in training.',
    },
    {
        'Technique': 'Weighted sampler',
        'Used in Experiment 3': 'No',
        'Purpose': 'Skipped intentionally to prioritize generalization-first training.',
    },
])
training_design_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_training_design.csv', index=False)
save_styled_table_figure(
    training_design_df,
    SAVE_DIRS['figures'] / 'experiment3_training_design.png',
    title='Experiment 3 Training Strategy',
    figsize=(17, 5.4),
    font_size=10,
)

phase_a_end = int(disease_history_df.loc[disease_history_df['phase'] == 'phase_a', 'epoch'].max())
best_epoch = int(summary['training_summary']['best_epoch'])

plt.figure(figsize=(11.5, 6.2))
ax = plt.gca()
ax.axvspan(1, phase_a_end, color=PRESENTATION_COLORS['gold'], alpha=0.12, label='Phase A: regularized training')
ax.axvspan(phase_a_end, disease_history_df['epoch'].max(), color=PRESENTATION_COLORS['teal'], alpha=0.10, label='Phase B: late fine-tune')
ax.plot(
    disease_history_df['epoch'],
    disease_history_df['selection_val_macro_f1'] * 100,
    color=PRESENTATION_COLORS['navy'],
    linewidth=2.4,
    label='Selected validation macro-F1',
)
ax.scatter(
    best_epoch,
    disease_history_df.loc[disease_history_df['epoch'] == best_epoch, 'selection_val_macro_f1'].iloc[0] * 100,
    color=PRESENTATION_COLORS['red'],
    s=60,
    zorder=4,
    label='Best epoch',
)
ax.set_title('Experiment 3 Validation Macro-F1 Across Training')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Macro-F1 (%)')
curve_values = disease_history_df['selection_val_macro_f1'] * 100
curve_floor = max(0, float(curve_values.min()) - 1.5)
curve_ceiling = min(100, float(curve_values.max()) + 1.0)
ax.set_ylim(curve_floor, curve_ceiling)
best_value = disease_history_df.loc[disease_history_df['epoch'] == best_epoch, 'selection_val_macro_f1'].iloc[0] * 100
ax.annotate(f'{best_value:.2f}', (best_epoch, best_value), xytext=(8, 8), textcoords='offset points', fontsize=9, color=PRESENTATION_COLORS['red'])
ax.grid(axis='y', linestyle='--', alpha=0.22)
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_training_curve.png', dpi=220)
plt.show()



## Section 5 - Raw Model Vs EMA Model


In [ ]:

ema_compare_df = pd.DataFrame([
    {
        'Model': 'Raw checkpoint',
        'Validation Accuracy (%)': percent(base_vs_ema_summary['raw_val']['acc']),
        'Validation Macro-F1 (%)': percent(base_vs_ema_summary['raw_val']['macro_f1']),
        'Test Accuracy (%)': percent(base_vs_ema_summary['raw_test']['acc']),
        'Test Macro-F1 (%)': percent(base_vs_ema_summary['raw_test']['macro_f1']),
    },
    {
        'Model': 'EMA checkpoint',
        'Validation Accuracy (%)': percent(base_vs_ema_summary['ema_val']['acc']),
        'Validation Macro-F1 (%)': percent(base_vs_ema_summary['ema_val']['macro_f1']),
        'Test Accuracy (%)': percent(base_vs_ema_summary['ema_test']['acc']),
        'Test Macro-F1 (%)': percent(base_vs_ema_summary['ema_test']['macro_f1']),
    },
])
ema_compare_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_raw_vs_ema.csv', index=False)
save_styled_table_figure(
    ema_compare_df,
    SAVE_DIRS['figures'] / 'experiment3_raw_vs_ema_table.png',
    title='Raw Checkpoint Vs EMA Checkpoint',
    figsize=(15, 4.2),
    font_size=10,
)

ema_plot_df = pd.DataFrame([
    ['Validation', 'Raw', percent(base_vs_ema_summary['raw_val']['macro_f1'])],
    ['Validation', 'EMA', percent(base_vs_ema_summary['ema_val']['macro_f1'])],
    ['Test', 'Raw', percent(base_vs_ema_summary['raw_test']['macro_f1'])],
    ['Test', 'EMA', percent(base_vs_ema_summary['ema_test']['macro_f1'])],
], columns=['Split', 'Model', 'Macro-F1 (%)'])

plt.figure(figsize=(9.5, 5.8))
ax = sns.barplot(
    data=ema_plot_df,
    x='Split',
    y='Macro-F1 (%)',
    hue='Model',
    palette=[PRESENTATION_COLORS['gold'], PRESENTATION_COLORS['teal']],
)
plt.title('EMA Selection Improved Validation Stability')
plt.xlabel('Split')
plt.ylabel('Macro-F1 (%)')
plt.ylim(90, 99)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=9, padding=3)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_raw_vs_ema_macro_f1.png', dpi=220)
plt.show()



## Section 6 - TTA Gains On The Selected Model


In [ ]:

tta_compare_df = pd.DataFrame([
    {
        'Split': 'Validation',
        'Base Accuracy (%)': percent(tta_summary['selected_base_val_acc']),
        'TTA Accuracy (%)': percent(tta_summary['tta_val_acc']),
        'Accuracy Gain (pts)': round(percent(tta_summary['tta_val_acc_gain']), 2),
        'Base Macro-F1 (%)': percent(tta_summary['selected_base_val_macro_f1']),
        'TTA Macro-F1 (%)': percent(tta_summary['tta_val_macro_f1']),
        'Macro-F1 Gain (pts)': round(percent(tta_summary['tta_val_macro_f1_gain']), 2),
    },
    {
        'Split': 'Test',
        'Base Accuracy (%)': percent(tta_summary['selected_base_test_acc']),
        'TTA Accuracy (%)': percent(tta_summary['tta_test_acc']),
        'Accuracy Gain (pts)': round(percent(tta_summary['tta_test_acc_gain']), 2),
        'Base Macro-F1 (%)': percent(tta_summary['selected_base_test_macro_f1']),
        'TTA Macro-F1 (%)': percent(tta_summary['tta_test_macro_f1']),
        'Macro-F1 Gain (pts)': round(percent(tta_summary['tta_test_macro_f1_gain']), 2),
    },
])
tta_compare_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_tta_gains.csv', index=False)
save_styled_table_figure(
    tta_compare_df,
    SAVE_DIRS['figures'] / 'experiment3_tta_gains_table.png',
    title='Full-Dataset TTA Gains',
    figsize=(17, 4.0),
    font_size=10,
)

tta_plot_df = pd.DataFrame([
    ['Validation Acc', percent(tta_summary['selected_base_val_acc']), percent(tta_summary['tta_val_acc'])],
    ['Validation Macro-F1', percent(tta_summary['selected_base_val_macro_f1']), percent(tta_summary['tta_val_macro_f1'])],
    ['Test Acc', percent(tta_summary['selected_base_test_acc']), percent(tta_summary['tta_test_acc'])],
    ['Test Macro-F1', percent(tta_summary['selected_base_test_macro_f1']), percent(tta_summary['tta_test_macro_f1'])],
], columns=['Metric', 'Base', 'TTA'])

x = np.arange(len(tta_plot_df))
width = 0.32
plt.figure(figsize=(11, 5.8))
ax = plt.gca()
base_bars = ax.bar(x - width / 2, tta_plot_df['Base'], width=width, color=PRESENTATION_COLORS['gray'], label='Base selected model')
tta_bars = ax.bar(x + width / 2, tta_plot_df['TTA'], width=width, color=PRESENTATION_COLORS['teal'], label='TTA model')
ax.set_xticks(x)
ax.set_xticklabels(tta_plot_df['Metric'], rotation=10)
ax.set_ylabel('Score (%)')
ax.set_title('TTA Raised Performance On Both Validation And Test Sets')
ax.set_ylim(90, 99)
ax.grid(axis='y', linestyle='--', alpha=0.22)
for bars in [base_bars, tta_bars]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_tta_gains.png', dpi=220)
plt.show()



## Section 7 - Experiment 3 Compared With Experiment 2


In [ ]:

comparison_row = exp2_vs_exp3_df.iloc[0]
exp2_vs_exp3_display_df = pd.DataFrame([
    {
        'System': 'Experiment 2 final pipeline',
        'Test Accuracy (%)': round(float(comparison_row['experiment2_test_acc']) * 100, 2),
        'Test Macro-F1 (%)': round(float(comparison_row['experiment2_test_macro_f1']) * 100, 2),
    },
    {
        'System': 'Experiment 3 base EMA model',
        'Test Accuracy (%)': round(float(comparison_row['experiment3_base_test_acc']) * 100, 2),
        'Test Macro-F1 (%)': round(float(comparison_row['experiment3_base_test_macro_f1']) * 100, 2),
    },
    {
        'System': 'Experiment 3 calibrated TTA',
        'Test Accuracy (%)': round(float(comparison_row['experiment3_calibrated_tta_test_acc']) * 100, 2),
        'Test Macro-F1 (%)': round(float(comparison_row['experiment3_calibrated_tta_test_macro_f1']) * 100, 2),
    },
])
exp2_vs_exp3_display_df.to_csv(SAVE_DIRS['tables'] / 'experiment2_vs_experiment3_comparison.csv', index=False)
save_styled_table_figure(
    exp2_vs_exp3_display_df,
    SAVE_DIRS['figures'] / 'experiment2_vs_experiment3_table.png',
    title='Experiment 2 Vs Experiment 3 Final Test Results',
    figsize=(15.5, 4.2),
    font_size=10,
)

plot_df = exp2_vs_exp3_display_df.copy()
x = np.arange(len(plot_df))
width = 0.34
plt.figure(figsize=(11, 5.8))
ax = plt.gca()
acc_bars = ax.bar(x - width / 2, plot_df['Test Accuracy (%)'], width=width, color=PRESENTATION_COLORS['gold'], label='Test Accuracy')
f1_bars = ax.bar(x + width / 2, plot_df['Test Macro-F1 (%)'], width=width, color=PRESENTATION_COLORS['teal'], label='Test Macro-F1')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['System'], rotation=12)
ax.set_ylabel('Score (%)')
ax.set_title('Experiment 3 Improved On The Experiment 2 Baseline')
ax.set_ylim(89, 96)
ax.grid(axis='y', linestyle='--', alpha=0.22)
for bars in [acc_bars, f1_bars]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.06, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment2_vs_experiment3_plot.png', dpi=220)
plt.show()



## Section 8 - Thresholds, Calibration, And Accepted-Only Results


In [ ]:

threshold_display_df = pd.DataFrame([
    {
        'Threshold': 'Leaf confidence',
        'Value': round(float(accepted_summary['min_leaf_confidence']), 2),
        'Meaning': 'Reject if the image does not confidently contain a tomato leaf.',
    },
    {
        'Threshold': 'Disease confidence',
        'Value': round(float(accepted_summary['min_disease_confidence']), 2),
        'Meaning': 'Reject if the top disease probability is too low.',
    },
    {
        'Threshold': 'Top-2 margin',
        'Value': round(float(accepted_summary['uncertainty_margin']), 2),
        'Meaning': 'Reject if the top disease is too close to the runner-up class.',
    },
    {
        'Threshold': 'Temperature',
        'Value': round(float(summary['temperature']), 3),
        'Meaning': 'Calibration factor learned from validation logits.',
    },
])
threshold_display_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_thresholds.csv', index=False)
save_styled_table_figure(
    threshold_display_df,
    SAVE_DIRS['figures'] / 'experiment3_thresholds_table.png',
    title='Experiment 3 Final Thresholds',
    figsize=(16, 4.4),
    font_size=10,
)

accepted_metrics_df = pd.DataFrame([
    {
        'Split': 'Validation accepted cases',
        'Coverage (%)': round(float(accepted_summary['validation_accepted_only']['coverage']) * 100, 2),
        'Reject Rate (%)': round(float(accepted_summary['validation_accepted_only']['reject_rate']) * 100, 2),
        'Accepted Accuracy (%)': round(float(accepted_summary['validation_accepted_only']['accepted_acc']) * 100, 2),
        'Accepted Macro-F1 (%)': round(float(accepted_summary['validation_accepted_only']['accepted_macro_f1']) * 100, 2),
    },
    {
        'Split': 'Test accepted cases',
        'Coverage (%)': round(float(accepted_summary['test_accepted_only']['coverage']) * 100, 2),
        'Reject Rate (%)': round(float(accepted_summary['test_accepted_only']['reject_rate']) * 100, 2),
        'Accepted Accuracy (%)': round(float(accepted_summary['test_accepted_only']['accepted_acc']) * 100, 2),
        'Accepted Macro-F1 (%)': round(float(accepted_summary['test_accepted_only']['accepted_macro_f1']) * 100, 2),
    },
])
accepted_metrics_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_accepted_metrics.csv', index=False)
save_styled_table_figure(
    accepted_metrics_df,
    SAVE_DIRS['figures'] / 'experiment3_accepted_metrics_table.png',
    title='Accepted-Only Results After Final Thresholding',
    figsize=(16, 4.0),
    font_size=10,
)

accepted_plot_df = pd.DataFrame([
    ['Coverage', round(float(accepted_summary['test_accepted_only']['coverage']) * 100, 2)],
    ['Reject rate', round(float(accepted_summary['test_accepted_only']['reject_rate']) * 100, 2)],
    ['Accepted accuracy', round(float(accepted_summary['test_accepted_only']['accepted_acc']) * 100, 2)],
    ['Accepted macro-F1', round(float(accepted_summary['test_accepted_only']['accepted_macro_f1']) * 100, 2)],
], columns=['Metric', 'Value'])

plt.figure(figsize=(9.8, 5.6))
ax = sns.barplot(
    data=accepted_plot_df,
    x='Metric',
    y='Value',
    hue='Metric',
    dodge=False,
    palette=[PRESENTATION_COLORS['gold'], PRESENTATION_COLORS['red'], PRESENTATION_COLORS['blue'], PRESENTATION_COLORS['teal']],
    legend=False,
)
plt.title('Final Thresholding Kept High Accuracy With Strong Coverage')
plt.xlabel('')
plt.ylabel('Percent (%)')
plt.ylim(0, 105)
for patch in ax.patches:
    value = patch.get_height()
    ax.annotate(f'{value:.2f}', (patch.get_x() + patch.get_width() / 2, value), ha='center', va='bottom', fontsize=9, xytext=(0, 5), textcoords='offset points')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_accepted_metrics_plot.png', dpi=220)
plt.show()



## Section 9 - Generalization Challenges And Error Analysis


In [ ]:

EXPERIMENT_2_WEAK_CLASSES = [
    'Pottassium_Deficiency',
    'Spotted_Wilt_Virus',
    'powdery_mildew',
    'Nitrogen_Deficiency',
    'Target_Spot',
]

split_shift_top_df = split_shift_df.head(8).copy()
split_shift_top_df['class_label'] = split_shift_top_df['class_name'].map(display_label)

plt.figure(figsize=(10.5, 5.8))
ax = sns.barplot(
    data=split_shift_top_df,
    y='class_label',
    x=split_shift_top_df['max_abs_share_delta'] * 100,
    hue='class_label',
    dodge=False,
    palette=[PRESENTATION_COLORS['red']] * len(split_shift_top_df),
    legend=False,
)
plt.title('Largest Class-Level Split Drift In The Dataset')
plt.xlabel('Maximum split-share drift (percentage points)')
plt.ylabel('Disease class')
for patch in ax.patches:
    value = patch.get_width()
    ax.annotate(f'{value:.2f}', (value, patch.get_y() + patch.get_height() / 2), ha='left', va='center', fontsize=9, xytext=(6, 0), textcoords='offset points')
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_split_shift_top8.png', dpi=220)
plt.show()

worst_classes_df = calibrated_per_class_df.head(8).copy()
worst_classes_df['true_class'] = worst_classes_df['true_class'].map(display_label)
worst_classes_df['per_class_accuracy_pct'] = worst_classes_df['per_class_accuracy'] * 100

plt.figure(figsize=(10.5, 6.0))
ax = sns.barplot(
    data=worst_classes_df,
    y='true_class',
    x='per_class_accuracy_pct',
    hue='true_class',
    dodge=False,
    palette=[PRESENTATION_COLORS['gold']] * len(worst_classes_df),
    legend=False,
)
plt.title('Lowest Per-Class Accuracy In Final Calibrated Test Predictions')
plt.xlabel('Per-class accuracy (%)')
plt.ylabel('Disease class')
plt.xlim(0, 100)
for patch in ax.patches:
    value = patch.get_width()
    ax.annotate(f'{value:.2f}', (value, patch.get_y() + patch.get_height() / 2), ha='left', va='center', fontsize=9, xytext=(6, 0), textcoords='offset points')
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_worst_classes.png', dpi=220)
plt.show()

weak_compare_df = per_class_compare_df[per_class_compare_df['true_class'].isin(EXPERIMENT_2_WEAK_CLASSES)].copy()
weak_compare_df['true_class'] = weak_compare_df['true_class'].map(display_label)
weak_compare_df['Experiment 2 Accuracy (%)'] = weak_compare_df['experiment2_per_class_accuracy'] * 100
weak_compare_df['Experiment 3 Accuracy (%)'] = weak_compare_df['experiment3_per_class_accuracy'] * 100
weak_compare_df['Delta (pts)'] = weak_compare_df['delta_accuracy'] * 100
weak_compare_display_df = weak_compare_df[[
    'true_class',
    'Experiment 2 Accuracy (%)',
    'Experiment 3 Accuracy (%)',
    'Delta (pts)',
]].rename(columns={'true_class': 'Weak class'})
weak_compare_display_df[['Experiment 2 Accuracy (%)', 'Experiment 3 Accuracy (%)', 'Delta (pts)']] = (
    weak_compare_display_df[['Experiment 2 Accuracy (%)', 'Experiment 3 Accuracy (%)', 'Delta (pts)']].round(2)
)
weak_compare_display_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_weak_class_comparison.csv', index=False)
save_styled_table_figure(
    weak_compare_display_df,
    SAVE_DIRS['figures'] / 'experiment3_weak_class_comparison.png',
    title='Experiment 2 Vs Experiment 3 On Important Weak Classes',
    figsize=(15.5, 4.2),
    font_size=10,
)

hardest_confusions_display_df = hardest_confusions_df.head(10).copy()
hardest_confusions_display_df['true_class'] = hardest_confusions_display_df['true_class'].map(display_label)
hardest_confusions_display_df['pred_class'] = hardest_confusions_display_df['pred_class'].map(display_label)
hardest_confusions_display_df = hardest_confusions_display_df.rename(columns={
    'true_class': 'True class',
    'pred_class': 'Predicted class',
    'count': 'Count',
})
hardest_confusions_display_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_hardest_confusions_top10.csv', index=False)
save_styled_table_figure(
    hardest_confusions_display_df,
    SAVE_DIRS['figures'] / 'experiment3_hardest_confusions_top10.png',
    title='Most Common Final-Test Confusions',
    figsize=(14.5, 4.6),
    font_size=10,
)



## Section 10 - Final Test Confusion Matrix


In [ ]:

plt.figure(figsize=(14, 12))
sns.heatmap(
    final_confusion_matrix,
    cmap='Blues',
    xticklabels=[display_label(name) for name in class_names],
    yticklabels=[display_label(name) for name in class_names],
    linewidths=0.15,
)
plt.title('Experiment 3 Final Test Confusion Matrix (Calibrated TTA)')
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'experiment3_final_confusion_matrix.png', dpi=220)
plt.show()



## Section 11 - Deployment Readiness And Sample Outputs


In [ ]:

deployment_summary_df = pd.DataFrame([
    {
        'Deployment detail': 'Selected inference model',
        'Value': summary['selected_model_for_inference'].upper(),
        'Why it matters': 'EMA was chosen as the final inference checkpoint.',
    },
    {
        'Deployment detail': 'TTA rounds',
        'Value': summary['tta_rounds'],
        'Why it matters': 'The final system averages seven inference views.',
    },
    {
        'Deployment detail': 'Preview accept rate',
        'Value': f"{deployment_preview_df['accepted'].mean() * 100:.2f}%",
        'Why it matters': 'All 40 sampled preview images were accepted in this saved run.',
    },
    {
        'Deployment detail': 'Leaf gate note',
        'Value': 'Synthetic proxy only',
        'Why it matters': 'The leaf gate still needs real greenhouse non-leaf validation.',
    },
])
deployment_summary_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_deployment_readiness.csv', index=False)
save_styled_table_figure(
    deployment_summary_df,
    SAVE_DIRS['figures'] / 'experiment3_deployment_readiness.png',
    title='Experiment 3 Deployment Readiness Notes',
    figsize=(16, 4.4),
    font_size=10,
)

sample_predictions_df = deployment_preview_df.copy()
sample_predictions_df['file_name'] = sample_predictions_df['image_path'].map(lambda x: Path(x).name)
sample_predictions_df['predicted_class'] = sample_predictions_df['predicted_class'].fillna('Rejected').map(display_label)
sample_predictions_df['top1_probability_pct'] = sample_predictions_df['top1_probability'] * 100
sample_predictions_df['margin_pct'] = sample_predictions_df['margin'] * 100
sample_predictions_display_df = sample_predictions_df[[
    'file_name', 'status', 'predicted_class', 'top1_probability_pct', 'margin_pct'
]].head(8).rename(columns={
    'file_name': 'Example file',
    'status': 'Status',
    'predicted_class': 'Predicted class',
    'top1_probability_pct': 'Top-1 probability (%)',
    'margin_pct': 'Top-2 margin (%)',
})
sample_predictions_display_df[['Top-1 probability (%)', 'Top-2 margin (%)']] = (
    sample_predictions_display_df[['Top-1 probability (%)', 'Top-2 margin (%)']].round(2)
)
sample_predictions_display_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_sample_predictions.csv', index=False)
save_styled_table_figure(
    sample_predictions_display_df,
    SAVE_DIRS['figures'] / 'experiment3_sample_predictions.png',
    title='Sample Deployment Predictions From Saved Test Preview',
    figsize=(17, 4.8),
    font_size=9,
)



## Section 12 - Final Takeaway


In [ ]:

final_takeaway_df = pd.DataFrame([
    {
        'Conclusion': 'Best final system',
        'Evidence': f"Experiment 3 reached {percent(summary['calibrated_tta_test_macro_f1']):.2f}% test macro-F1 with calibrated TTA.",
    },
    {
        'Conclusion': 'Improvement over Experiment 2',
        'Evidence': f"The final Experiment 3 system improved test macro-F1 by {percent(summary['experiment2_comparison']['delta_calibrated_tta_test_macro_f1_vs_exp2']):.2f} percentage points.",
    },
    {
        'Conclusion': 'High-confidence deployment mode',
        'Evidence': f"After thresholding, accepted test predictions reached {percent(summary['accepted_only_test_macro_f1']):.2f}% macro-F1 at {percent(summary['accepted_only_test_coverage']):.2f}% coverage.",
    },
    {
        'Conclusion': 'Main caveat',
        'Evidence': 'The leaf gate remains a synthetic proxy and should later be validated with real non-leaf greenhouse images.',
    },
])
final_takeaway_df.to_csv(SAVE_DIRS['tables'] / 'experiment3_final_takeaway.csv', index=False)
save_styled_table_figure(
    final_takeaway_df,
    SAVE_DIRS['figures'] / 'experiment3_final_takeaway.png',
    title='Experiment 3 Final Takeaway',
    figsize=(16, 4.6),
    font_size=10,
)

print('Experiment 3 presentation notebook is ready.')
print('Selected inference model    :', summary['selected_model_for_inference'].upper())
print('Final calibrated test acc   :', f"{percent(summary['calibrated_tta_test_acc']):.2f}%")
print('Final calibrated test macro-F1:', f"{percent(summary['calibrated_tta_test_macro_f1']):.2f}%")
print('Accepted-only test macro-F1 :', f"{percent(summary['accepted_only_test_macro_f1']):.2f}%")
print('Accepted-only test coverage :', f"{percent(summary['accepted_only_test_coverage']):.2f}%")
print('Presentation output root    :', PRESENTATION_ROOT)
